<a href="https://colab.research.google.com/github/Tseng0318/peft/blob/main/Model_Merging_GT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model Merging — Inference with Ground Truth
Uses `allenai/ai2_arc` (ARC-Challenge) for science and `cais/mmlu` (high_school_mathematics) for math.
Both datasets are MCQA with ground truth answer keys, so accuracy can be computed directly.

In [1]:
!nvidia-smi

Sat Apr 18 13:06:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             46W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Install Packages

In [2]:
!pip install -q --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128
!pip install -q datasets==3.5.1 bitsandbytes>=0.46.1 huggingface_hub>=1.3.0

## Import Packages

In [3]:
%env CUBLAS_WORKSPACE_CONFIG=:4096:8

import json, os, re
import numpy as np
import torch
from tqdm import tqdm
from datasets import load_dataset
from peft import PeftModel, LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig, BitsAndBytesConfig

env: CUBLAS_WORKSPACE_CONFIG=:4096:8


## Load Datasets with Ground Truth
- **Math**: `cais/mmlu` → `high_school_mathematics` test split (100 questions, MCQA, answerKey 0-3)
- **Science**: `allenai/ai2_arc` → `ARC-Challenge` test split (MCQA, answerKey A-D)

In [4]:
N_SAMPLES = 200  # set to None to use the full test split

# ── Math: MMLU high school mathematics ──
mmlu_raw = load_dataset("cais/mmlu", "high_school_mathematics", split="test")
if N_SAMPLES:
    mmlu_raw = mmlu_raw.select(range(min(N_SAMPLES, len(mmlu_raw))))

OPTION_LABELS = ["A", "B", "C", "D"]
math_data = []
for i, item in enumerate(mmlu_raw):
    math_data.append({
        "id": f"mmlu_math_{i}",
        "question": item["question"],
        "options": {OPTION_LABELS[j]: item["choices"][j] for j in range(4)},
        "answer": OPTION_LABELS[item["answer"]],  # ground truth
    })

# ── Science: ARC-Challenge ──
arc_raw = load_dataset("allenai/ai2_arc", "ARC-Challenge", split="test")
if N_SAMPLES:
    arc_raw = arc_raw.select(range(min(N_SAMPLES, len(arc_raw))))

science_data = []
for i, item in enumerate(arc_raw):
    labels = item["choices"]["label"]
    texts  = item["choices"]["text"]
    # Normalize labels: some ARC items use 1/2/3/4 instead of A/B/C/D
    norm = {"1":"A","2":"B","3":"C","4":"D"}
    options = {norm.get(l, l): t for l, t in zip(labels, texts)}
    ak = item["answerKey"]
    ak = norm.get(ak, ak)
    science_data.append({
        "id": f"arc_{i}",
        "question": item["question"],
        "options": options,
        "answer": ak,  # ground truth
    })

print(f"Math (MMLU)   : {len(math_data)} questions")
print(f"Science (ARC) : {len(science_data)} questions")
print(f"\nSample Math   : {math_data[0]}")
print(f"\nSample Science: {science_data[0]}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

high_school_mathematics/test-00000-of-00(…):   0%|          | 0.00/33.7k [00:00<?, ?B/s]

high_school_mathematics/validation-00000(…):   0%|          | 0.00/6.99k [00:00<?, ?B/s]

...

ARC-Challenge/validation-00000-of-00001.(…):   0%|          | 0.00/55.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1119 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1172 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/299 [00:00<?, ? examples/s]

Math (MMLU)   : 200 questions
Science (ARC) : 200 questions

Sample Math   : {'id': 'mmlu_math_0', 'question': 'If a pentagon P with vertices at (– 2, – 4), (– 4, 1), (–1, 4), (2, 4), and (3, 0) is reflected across the line y = x to get a new pentagon, P’, then one of the vertices of P’ is', 'options': {'A': '(0, – 3)', 'B': '(4, 1)', 'C': '(2, 2)', 'D': '(– 4, –2)'}, 'answer': 'D'}

Sample Science: {'id': 'arc_0', 'question': 'An astronomer observes that a planet rotates faster after a meteorite impact. Which is the most likely effect of this increase in rotation?', 'options': {'A': 'Planetary density will decrease.', 'B': 'Planetary years will become longer.', 'C': 'Planetary days will become shorter.', 'D': 'Planetary gravity will become stronger.'}, 'answer': 'C'}


## Save Datasets Locally

In [15]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/merge_result"
os.makedirs(DRIVE_DIR, exist_ok=True)

with open(f"{DRIVE_DIR}/MATH.json", "w") as f:
    json.dump(math_data, f, indent=2)
with open(f"{DRIVE_DIR}/SCIENCE.json", "w") as f:
    json.dump(science_data, f, indent=2)
print(f"Saved MATH.json and SCIENCE.json → {DRIVE_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Saved MATH.json and SCIENCE.json → /content/drive/MyDrive/merge_result


## Prompt Generation

In [16]:
INSTRUCTION = {
    "MATH":    "You are given a math question and four answer options (associated with \"A\", \"B\", \"C\", \"D\"). Your task is to carefully analyze the problem, apply logical reasoning, and select the correct answer. There is only one correct answer for each question.",
    "SCIENCE": "You are given a science question and four answer options (associated with \"A\", \"B\", \"C\", \"D\"). Your task is to find the correct answer based on scientific facts, knowledge, and reasoning. There is only one correct answer for each question.",
}

def build_prompt(task_name, data_point):
    sys_msg = INSTRUCTION[task_name]
    opts = data_point["options"]
    options_str = " ".join([f"({k}) {v}" for k, v in opts.items()])
    user_prompt = f"Question: {data_point['question']}; Options: {options_str}"
    return f"""[INST] <<SYS>>You are a helpful assistant and good at solving tasks based on task instructions and inputs.<</SYS>> Instruction: {sys_msg} Input: {user_prompt}[/INST]"""

## Seed & Device Settings

In [17]:
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device = torch.device("cuda:0")
torch.manual_seed(42)
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
np.random.seed(42)

## Merging Settings

In [31]:
#### adjust ####
MERGE_TYPE   = "linear"   # linear | magnitude_prune | ties | dare_ties | dare_linear
density      = 0.2
weight_math  = 1.0
weight_sci   = 0.4
weights      = [weight_math, weight_sci]

OUTPUT_DIR = "/content/drive/MyDrive/merge_result/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

## Load Model & Adapters

In [32]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True
)
lora_config = LoraConfig(
    r=8, target_modules=["q_proj","k_proj","v_proj"],
    task_type="CAUSAL_LM", lora_alpha=16, lora_dropout=0.05
)

base_model_name = "unsloth/llama-2-7b-chat-bnb-4bit"
tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True)
tokenizer.pad_token_id = 1
tokenizer.eos_token_id = 2

model = AutoModelForCausalLM.from_pretrained(
    base_model_name, quantization_config=quant_config,
    low_cpu_mem_usage=True, torch_dtype=torch.float16
).to(device)
model.resize_token_embeddings(len(tokenizer))

hf_adapters = {
    "MATH":    "MonicaHuang/llama-2-7b-chat-GSM8K-MCQA",
    "SCIENCE": "chenjoachim/llama-2-7b-chat-ARC-MCQA",
}
TASK_NAMES = ["MATH", "SCIENCE"]

model = PeftModel.from_pretrained(model, hf_adapters[TASK_NAMES[0]], adapter_name=TASK_NAMES[0]).to(device)
for t in TASK_NAMES[1:]:
    model.load_adapter(hf_adapters[t], adapter_name=t, device=device)

print("Model and adapters loaded.")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model and adapters loaded.


## Apply Merging Algorithm

In [33]:
if MERGE_TYPE == "ties":
    model.add_weighted_adapter(TASK_NAMES, weights, "merge", combination_type="ties", density=density)
elif MERGE_TYPE == "linear":
    model.add_weighted_adapter(TASK_NAMES, weights, "merge", combination_type="linear")
elif MERGE_TYPE == "magnitude_prune":
    model.add_weighted_adapter(TASK_NAMES, weights, "merge", combination_type="magnitude_prune", density=density)
elif MERGE_TYPE == "dare_ties":
    model.add_weighted_adapter(TASK_NAMES, weights, "merge", combination_type="dare_ties", density=density)
elif MERGE_TYPE == "dare_linear":
    model.add_weighted_adapter(TASK_NAMES, weights, "merge", combination_type="dare_linear", density=density)

model.set_adapter("merge")
print(f"Merged with: {MERGE_TYPE} | density={density} | weights={weights}")

Merged with: linear | density=0.2 | weights=[1.0, 0.4]


## Generation Config

In [34]:
hyperparameters = {
    "do_sample": False, "temperature": None,
    "num_beams": 1, "top_p": None,
    "no_repeat_ngram_size": 3, "max_new_len": 400
}
generation_config = GenerationConfig(
    do_sample=hyperparameters["do_sample"],
    temperature=hyperparameters["temperature"],
    top_p=hyperparameters["top_p"],
    num_beams=hyperparameters["num_beams"],
    pad_token_id=1,
    max_new_tokens=hyperparameters["max_new_len"]
)
print("Generation config set.")

Generation config set.


## Inference Helper

In [35]:
def run_inference(task_name, test_datas):
    results = []
    for data_point in tqdm(test_datas, desc=task_name):
        prompt = build_prompt(task_name, data_point)
        inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=True)
        input_ids = inputs["input_ids"].cuda()

        model.eval()
        with torch.no_grad():
            output = model.generate(
                input_ids=input_ids,
                generation_config=generation_config,
                return_dict_in_generate=True,
                output_scores=True,
                max_new_tokens=hyperparameters["max_new_len"],
            )
        for s in output.sequences:
            predict = tokenizer.decode(s)
        response = predict.split("[/INST]")[-1].split("</s>")[0].strip()

        results.append({
            "id":       data_point["id"],
            "response": response,
            "answer":   data_point["answer"],   # ground truth stored alongside
        })
    return results

## Run Math Inference (MMLU)

In [36]:
wnames = '_'.join([str(float(w)) for w in weights])

In [37]:
math_results = run_inference("MATH", math_data)

result_dir = f"{OUTPUT_DIR}/{MERGE_TYPE}/MATH"
os.makedirs(result_dir, exist_ok=True)
out_path = f"{result_dir}/w_{wnames}_d_{density}_result.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(math_results, f, indent=2, ensure_ascii=False)
print(f"Saved → {out_path}")

MATH: 100%|██████████| 200/200 [26:44<00:00,  8.02s/it]

Saved → /content/drive/MyDrive/merge_result/outputs/linear/MATH/w_1.0_0.4_d_0.2_result.json


## Run Science Inference (ARC-Challenge)

In [38]:
science_results = run_inference("SCIENCE", science_data)



SCIENCE: 100%|██████████| 200/200 [19:46<00:00,  5.93s/it]


In [39]:
result_dir = f"{OUTPUT_DIR}/{MERGE_TYPE}/SCIENCE"
os.makedirs(result_dir, exist_ok=True)
out_path = f"{result_dir}/w_{wnames}_d_{density}_result.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(science_results, f, indent=2, ensure_ascii=False)
print(f"Saved → {out_path}")

Saved → /content/drive/MyDrive/merge_result/outputs/linear/SCIENCE/w_1.0_0.4_d_0.2_result.json


## MATH adapter baseline:

In [27]:
model.set_adapter("MATH")

math_only = run_inference("MATH", math_data)

result_dir = f"{OUTPUT_DIR}/MATH_only/MATH"
os.makedirs(result_dir, exist_ok=True)
with open(f"{result_dir}/w_1.0_d_1.0_result.json", "w") as f:
    json.dump(math_only, f, indent=2)

MATH:   0%|          | 0/200 [00:06<?, ?it/s]


KeyboardInterrupt: 

## SCIENCE adapter baseline

In [17]:
model.set_adapter("SCIENCE")

science_only = run_inference("SCIENCE", science_data)

result_dir = f"{OUTPUT_DIR}/SCIENCE_only/SCIENCE"
os.makedirs(result_dir, exist_ok=True)
with open(f"{result_dir}/w_1.0_d_1.0_result.json", "w") as f:
    json.dump(science_only, f, indent=2)

SCIENCE: 100%|██████████| 200/200 [03:44<00:00,  1.12s/it]


## Compute Accuracy

In [40]:
def extract_answer(response):
    matches = re.findall(r'\(([A-D])\)', response)
    return matches[-1] if matches else None

def accuracy(results):
    correct = no_pred = 0
    for r in results:
        pred = extract_answer(r["response"])
        if pred is None:
            no_pred += 1
        correct += (pred == r["answer"])
    total = len(results)
    return correct / total, correct, total, no_pred

def load(path):
    with open(path) as f:
        return json.load(f)

In [43]:
OUTPUT_DIR = "/content/drive/MyDrive/merge_result/outputs"
MERGE_TYPE = "linear"
density    = 0.2
weights    = [1.0, 0.4]
wnames     = '_'.join([str(float(w)) for w in weights])

math_results  = load(f"{OUTPUT_DIR}/{MERGE_TYPE}/MATH/w_{wnames}_d_{density}_result.json")
sci_results   = load(f"{OUTPUT_DIR}/{MERGE_TYPE}/SCIENCE/w_{wnames}_d_{density}_result.json")
math_only     = load(f"{OUTPUT_DIR}/MATH_only/MATH/w_1.0_d_1.0_result.json")
science_only  = load(f"{OUTPUT_DIR}/SCIENCE_only/SCIENCE/w_1.0_d_1.0_result.json")

math_acc,      mc, mt, mn = accuracy(math_results)
sci_acc,       sc, st, sn = accuracy(sci_results)
math_only_acc,  _,  _,  _ = accuracy(math_only)
sci_only_acc,   _,  _,  _ = accuracy(science_only)
avg = (math_acc + sci_acc) / 2




In [44]:
print(f"{'':25s} {'MATH':>8} {'SCIENCE':>10}")
print("-" * 45)
print(f"{'MATH adapter only':25s} {math_only_acc:>8.2%} {'N/A':>10}")
print(f"{'SCIENCE adapter only':25s} {'N/A':>8} {sci_only_acc:>10.2%}")
print(f"{'Merged (' + MERGE_TYPE + ')':25s} {math_acc:>8.2%} {sci_acc:>10.2%}")
print(f"{'Average (merged)':25s} {avg:>8.2%}")

                              MATH    SCIENCE
---------------------------------------------
MATH adapter only           22.00%        N/A
SCIENCE adapter only           N/A     62.00%
Merged (linear)             15.00%     23.50%
Average (merged)            19.25%


```
                              MATH    SCIENCE
---------------------------------------------
MATH adapter only           22.00%        N/A
SCIENCE adapter only           N/A     62.00%
Merged (dare_ties)          14.50%     43.00%
Average (merged)            28.75%
```

```
                              MATH    SCIENCE
---------------------------------------------
MATH adapter only           22.00%        N/A
SCIENCE adapter only           N/A     62.00%
Merged (magnitude_prune)    26.50%     48.50%
Average (merged)            37.50%
```


```
                              MATH    SCIENCE
---------------------------------------------
MATH adapter only           22.00%        N/A
SCIENCE adapter only           N/A     62.00%
Merged (linear)             15.00%     23.50%
Average (merged)            19.25%
```
